In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [ ]:
import mlflow
import os
db_path = os.path.abspath("/workspaces/MLOPS/00-mlflow/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nyc-taxi-exp")

#mlflow ui --backend-store-uri sqlite:///mlflow.db

2026/09/08 12:55:33 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-exp' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/MLOPS/03-Experiment_tracking/mlruns/2', creation_time=1788872133033, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1788872133033, lifecycle_stage='active', name='nyc-taxi-exp', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_dataframe(path):
    df=pd.read_parquet(path)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime-df.lpep_pickup_datetime
    
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df=df[((df.duration>=1) & (df.duration<=60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    
    df[categorical]=df[categorical].astype(str)

    return df

In [4]:
df_train=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet')
df_val=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet')

In [5]:
len(df_train), len(df_val)

(38088, 35319)

In [6]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv=DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [7]:
target='duration'
y_train= df_train[target].values
y_val= df_val[target].values

In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

8.33683572243964

In [9]:
X_train.indices = X_train.indices.astype('int32')
X_train.indptr = X_train.indptr.astype('int32')

X_val.indices = X_val.indices.astype('int32')
X_val.indptr = X_val.indptr.astype('int32')

with mlflow.start_run():
    mlflow.set_tag("developer", "aitor")
    
    mlflow.log_param("train-data-path","https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet")
    mlflow.log_param("valid-data-path","https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet")
    
    alpha=0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse=root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [10]:
train=xgb.DMatrix(X_train, label=y_train)
valid=xgb.DMatrix(X_val, label=y_val)

def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50,
            verbose_eval=100
        )
        y_pred=booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        
    return {'loss':rmse, 'status': STATUS_OK}

In [11]:
search_space={
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42,
}

best_result = fmin(fn=objective,
                   space=search_space,
                   algo=tpe.suggest,
                   max_evals=50,
                   trials=Trials(),
                   verbose=False)

[0]	validation-rmse:7.22996
[57]	validation-rmse:6.35083
[0]	validation-rmse:6.42125
[51]	validation-rmse:6.53694
[0]	validation-rmse:9.92420
[100]	validation-rmse:5.96535
[129]	validation-rmse:5.97128
[0]	validation-rmse:9.83064
[100]	validation-rmse:5.80465
[200]	validation-rmse:5.78272
[267]	validation-rmse:5.78333
[0]	validation-rmse:6.41097
[51]	validation-rmse:7.00755
[0]	validation-rmse:9.47811
[70]	validation-rmse:6.39784
[0]	validation-rmse:10.02096
[100]	validation-rmse:6.12402
[102]	validation-rmse:6.12515
[0]	validation-rmse:6.81450
[50]	validation-rmse:6.98076
[0]	validation-rmse:8.17501
[56]	validation-rmse:6.53669
[0]	validation-rmse:10.00066
[95]	validation-rmse:6.21089
[0]	validation-rmse:8.43970
[68]	validation-rmse:5.87287
[0]	validation-rmse:8.40371
[56]	validation-rmse:6.40046
[0]	validation-rmse:9.31479
[100]	validation-rmse:5.82607
[200]	validation-rmse:5.80156
[287]	validation-rmse:5.80563
[0]	validation-rmse:9.51269
[86]	validation-rmse:5.98655
[0]	validation-r

In [13]:
mlflow.xgboost.autolog(disable=False)

In [14]:
with mlflow.start_run():
    best_params={
    "learning_rate": 0.07514278112841959,
    "max_depth": 6,
    "min_child_weight": 9.028786614533509,
    "objective": "reg:squarederror",
    "reg_alpha": 0.04080737438776132,
    "reg_lambda": 0.04628904701264794,
    "seed":42,
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
                params=best_params,
                dtrain=train,
                num_boost_round=1000,
                evals=[(valid, "validation")],
                early_stopping_rounds=50,
                verbose_eval=100
            )

    y_pred = booster.predict(valid)
    rmse=root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse",rmse)
    
    
    with open("models/preprocessor.b","wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    
    mlflow.xgboost.log_model(booster, name="models_mlflow")
    

[0]	validation-rmse:9.88789
[100]	validation-rmse:5.91456
[200]	validation-rmse:5.84694
[300]	validation-rmse:5.81466
[400]	validation-rmse:5.79684
[500]	validation-rmse:5.78252
[600]	validation-rmse:5.77501
[700]	validation-rmse:5.76980
[800]	validation-rmse:5.76823
[900]	validation-rmse:5.76585
[938]	validation-rmse:5.76572


2026/09/08 13:03:01 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/09/08 13:03:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [15]:
# Replace with your actual model URI (e.g., "runs:/<run_id>/my_model" or "models:/<model_name>/1")
model_uri = "runs:/cd2e03335b6f4450b98eafc7fa89a117/models_mlflow"
loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(valid)
rmse=root_mean_squared_error(y_val, y_pred)
print(rmse)

5.765722147877208
